In [ ]:
import numpy as np
import torch
import random
import pandas as pd

from datasets import load_dataset, Audio

from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    AutoConfig,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model_id = "facebook/mms-300m"

feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id,
    do_normalize=True,
    return_attention_mask=True
)

dataset = load_dataset("badrex/nnti-dataset-full")

train_ds = dataset["train"].shuffle(seed=42)
valid_ds = dataset["validation"].shuffle(seed=42)

train_ds = train_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))
valid_ds = valid_ds.cast_column("audio_filepath", Audio(sampling_rate=16000))

LABELS = train_ds.unique("language")

str_to_int = {s: i for i, s in enumerate(LABELS)}
int_to_str = {i: s for s, i in str_to_int.items()}

#Audio augmentation- bias mitigation

def augment_audio(audio):

    # random volume scaling
    if random.random() < 0.5:
        audio = audio * np.random.uniform(0.8, 1.2)

    # gaussian noise
    if random.random() < 0.3:
        noise = np.random.normal(0, 0.005, audio.shape)
        audio = audio + noise

    return audio

max_duration = 7

def preprocess_train(batch):

    audio_arrays = []

    for x in batch["audio_filepath"]:

        audio = x["array"]

        audio = augment_audio(audio)

        audio_arrays.append(audio)

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        truncation=True,
        max_length=int(16000 * max_duration),
        return_attention_mask=True
    )

    inputs["label"] = [str_to_int[x] for x in batch["language"]]

    return inputs

def preprocess_val(batch):

    audio_arrays = [x["array"] for x in batch["audio_filepath"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        truncation=True,
        max_length=int(16000 * max_duration),
        return_attention_mask=True
    )

    inputs["label"] = [str_to_int[x] for x in batch["language"]]

    return inputs

train_ds = train_ds.map(
    preprocess_train,
    remove_columns=train_ds.column_names,
    batched=True
)

valid_ds = valid_ds.map(
    preprocess_val,
    remove_columns=valid_ds.column_names,
    batched=True
)

config = AutoConfig.from_pretrained(model_id)

config.num_labels = len(LABELS)
config.label2id = str_to_int
config.id2label = int_to_str

model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    config=config
)

class DataCollator:

    def __init__(self, fe):
        self.fe = fe

    def __call__(self, features):

        batch = {
            "input_values": [f["input_values"] for f in features],
            "attention_mask": [f["attention_mask"] for f in features]
        }

        batch = self.fe.pad(batch, return_tensors="pt")

        batch["labels"] = torch.tensor([f["label"] for f in features])

        return batch


data_collator = DataCollator(feature_extractor)

training_args = TrainingArguments(

    output_dir="./augmentation_results",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    num_train_epochs=6,

    learning_rate=1e-5,

    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    fp16=True,

    report_to="none"
)

def compute_metrics(eval_pred):

    preds = np.argmax(eval_pred.predictions, axis=1)

    acc = accuracy_score(eval_pred.label_ids, preds)

    return {"accuracy": acc}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Training with augmentation...")

trainer.train()

trainer.evaluate()

trainer.save_model("/kaggle/working/mms_augmented_model")

#Analysis Section

#Predictions
predictions = trainer.predict(valid_ds)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

#Confusion Matrix

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12,10))

sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=int_to_str.values(),
    yticklabels=int_to_str.values()
)

plt.xlabel("Predicted")
plt.ylabel("True")

plt.title("Confusion Matrix")

plt.savefig("/kaggle/working/augmentation_confusion_matrix.png")

plt.show()

#t-SNE Visualization

from sklearn.manifold import TSNE

embeddings = predictions.predictions

tsne = TSNE(n_components=2, random_state=42)

X = tsne.fit_transform(embeddings)

plt.figure(figsize=(8,6))

plt.scatter(
    X[:,0],
    X[:,1],
    c=y_true,
    cmap="tab20",
    s=5
)

plt.title("t-SNE of Language Embeddings")

plt.savefig("/kaggle/working/tsne_embeddings.png")

plt.show()

#Training curves

logs = trainer.state.log_history

log_df = pd.DataFrame(logs)

log_df.to_csv("/kaggle/working/augmentation_logs.csv")

#Loss Curves

train_loss = log_df["loss"].dropna()

plt.plot(train_loss)

plt.title("Training Loss")

plt.savefig("/kaggle/working/augmentation_loss.png")

plt.show()


GPU available: False


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/382M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/383M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training with augmentation...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
